In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import torch
device= "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

# Text Cleaning


In [3]:
import re
train= pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
print(train.isnull().sum().sum())  #no nulls values

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\-\./\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["cleaned_prompt"] = train["prompt"].apply(clean_text)
for opt in ["A","B","C","D","E"]:
    train[f"cleaned_{opt}"] = train[f"{opt}"].apply(clean_text)
train.head(5)

0


,id,prompt,A,B,C,D,E,answer,cleaned_prompt,cleaned_A,cleaned_B,cleaned_C,cleaned_D,cleaned_E
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,pick the best possible answer what is martin h...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,what is accelerator-based light-ion fusion,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,determine the correct option what is the term ...,blueshifting,redshifting,reddening,whitening,yellowing
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,select the most accurate option what is martin...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,identify the correct statement what is the con...,simultaneity is relative meaning that two even...,simultaneity is relative meaning that two even...,simultaneity is absolute meaning that two even...,simultaneity is a concept that applies only to...,simultaneity is a concept that applies only to...


In [4]:
train['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

# Baseline Model


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(train, test_size=0.2,random_state=42, stratify=train["answer"])

texts_train = []
labels_train = []
texts_test = []
labels_test = []
options = ["A","B","C","D","E"]

for i in range(len(train_df)):
    prompt = train_df.iloc[i]["cleaned_prompt"]
    answer = train_df.iloc[i]["answer"]
    for opt in options:
        text = prompt + " " + train_df.iloc[i][f"cleaned_{opt}"]
        texts_train.append(text)
        labels_train.append(1 if answer == opt else 0)
for i in range(len(test_df)):
    prompt = test_df.iloc[i]["cleaned_prompt"]
    answer = test_df.iloc[i]["answer"]
    for opt in options:
        text = prompt + " " + test_df.iloc[i][f"cleaned_{opt}"]
        texts_test.append(text)
        labels_test.append(1 if answer == opt else 0)
        
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(texts_train)
X_test_tfidf = tfidf.transform(texts_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, labels_train)
y_pred = lr.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(labels_test, y_pred))
print("F1 Score:", f1_score(labels_test, y_pred))

probs = lr.predict_proba(X_test_tfidf)[:,1]
rows_per_q = 5
pred_opt = []
true_ans = []
for i in range(0, len(probs), rows_per_q):
    q_probs = probs[i:i+5]
    ranked = np.argsort(-q_probs)
    pred_opt.append([options[j] for j in ranked[:3]])
    correct_idx = np.argmax(labels_test[i:i+5])
    true_ans.append(options[correct_idx])

def map3(true_answers, predicted_options, k=3):
    score = 0
    for actual, preds in zip(true_answers, predicted_options):
        if actual in preds[:k]:
            score += 1.0 / (preds.index(actual) + 1)
    return score / len(true_answers)
map3_score = map3(true_ans, pred_opt)
print("MAP@3:", map3_score)

Accuracy: 0.813
F1 Score: 0.12206572769953052
MAP@3: 0.8787499999999997


In [6]:
!pip install wandb

In [7]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("wandb")
wandb.login()

In [8]:
wandb.init(
    project="dlgenai-t226",
    name="baseline-model",
    config={"Model": "Logistic Regression",
            "Random Seed": 42,
            "Train/Test Split": "80/20" })
wandb.log({"accuracy": accuracy_score(labels_test, y_pred),
            "f1_score": f1_score(labels_test, y_pred),
            "map@3": map3_score })
wandb.finish()

# RoBERTa tokenization & training

In [9]:
import random
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
torch.cuda.manual_seed_all(42)

In [10]:
from transformers import AutoTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

def create_pairs(df):
    rows = []
    for _, row in df.iterrows():
        prompt = row["cleaned_prompt"]
        for opt in options:
            rows.append({"question_id": row["id"],
                    "text": prompt + " </s> " + row[f"cleaned_{opt}"],
                    "label": 1 if row["answer"] == opt else 0, "option": opt})
    return pd.DataFrame(rows)
train_pairs = create_pairs(train_df)
test_pairs = create_pairs(test_df)
train_dataset = Dataset.from_pandas(train_pairs)
val_dataset = Dataset.from_pandas(test_pairs)

In [11]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True,padding="max_length",max_length=256 )
    
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids","attention_mask","label"])
val_dataset.set_format(type="torch", columns=["input_ids","attention_mask","label"])

model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
training_args = TrainingArguments(output_dir="./roberta_results",
                            eval_strategy="epoch",
                            save_strategy="epoch",
                            learning_rate=2e-5,
                            per_device_train_batch_size=16,
                            per_device_eval_batch_size=16,
                            num_train_epochs=3,
                            weight_decay=0.01,
                            logging_steps=100,
                            load_best_model_at_end=True, seed=42)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=val_dataset)
trainer.train()

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=750, training_loss=0.821944943745931, metrics={'train_runtime': 669.1153, 'train_samples_per_second': 35.868, 'train_steps_per_second': 1.121, 'total_flos': 3157332664320000.0, 'train_loss': 0.821944943745931, 'epoch': 3.0})

In [12]:
preds = trainer.predict(val_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Accuracy:", acc)
print("F1 Score:", f1)

probs = torch.softmax(torch.tensor(preds.predictions), dim=1).numpy()
positive_probs = probs[:, 1]
rows_per_q = 5
pred_opt = []
true_ans = []

def map3(actual, predicted):
    score = 0
    for a, p in zip(actual, predicted):
        if a in p:
            score += 1 / (p.index(a) + 1)
    return score / len(actual)
    
for i in range(0, len(positive_probs), rows_per_q):
    q_probs = positive_probs[i:i+5]
    ranked_indices = np.argsort(q_probs)[::-1]
    top3 = [options[j] for j in ranked_indices[:3]]
    pred_opt.append(top3)
    correct_index = np.argmax(y_true[i:i+5])
    true_ans.append(options[correct_index])

map3_r = map3(true_ans, pred_opt)
print("MAP@3:", map3_r)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Accuracy: 0.916
F1 Score: 0.7633802816901408
MAP@3: 0.9429166666666666


In [13]:
wandb.init(project="dlgenai-t226",
            name="roberta-pretrained",
           config={"Model": "RoBERTa",
                    "Epochs": 3,
                    "Batch Size": 16,
                    "Learning Rate": 2e-5} )
wandb.log({ "accuracy": acc,
            "f1_score": f1,
             "map@3": map3_r})
wandb.finish()

# Sentence transformer

In [20]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")
pred_opt = []
true_ans = []

for _, row in test_df.iterrows():
    prompt = row["cleaned_prompt"]
    opt_texts = [row["cleaned_A"], row["cleaned_B"],row["cleaned_C"],row["cleaned_D"], row["cleaned_E"] ]
    prompt_embd = model.encode([prompt])
    opt_embd = model.encode(opt_texts)
    scores = cosine_similarity(prompt_embd, opt_embd)[0]
    ranked = np.argsort(scores)[::-1]
    top3 = [options[i] for i in ranked[:3]]
    pred_opt.append(top3)
    true_ans.append(row["answer"])

map3_s = map3(true_ans, pred_opt)
print("MAP@3:", map3_s)

opt1 = [p[0] for p in pred_opt]
acc_s = accuracy_score(true_ans, opt1)
print("Accuracy:", acc_s)

f1_s = f1_score(true_ans,opt1, average="macro")
print("F1 Score:", f1_s)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MAP@3: 0.3825
Accuracy: 0.225
F1 Score: 0.22427065954133768


In [ ]:
wandb.init( project="dlgenai-t226",
            name="SentenceTransformer",
            config={"Model": "Sentence Transformer",
                    "Embedding Model": "all-MiniLM-L6-v2",
                    "Similarity Metric": "Cosine Similarity"})
wandb.log({"accuracy": acc,
            "f1_score": f1,
            "map@3": map3_s})
wandb.finish()

# Models comparison

In [21]:
print("TFIDF + Logistic Regression:")
print("    Accuracy:", accuracy_score(labels_test, y_pred))
print("    F1:", f1_score(labels_test, y_pred))
print("    MAP@3:", map3_score)

print("Sentence Transformer:")
print("    Accuracy:", acc_s)
print("    F1:", f1_s)
print("    MAP@3:", map3_s)

print("RoBERTa:")
print("    Accuracy:", acc)
print("    F1:", f1)
print("    MAP@3:", map3_r)

TFIDF + Logistic Regression:
    Accuracy: 0.916
    F1: 0.7633802816901408
    MAP@3: 0.8787499999999997
Sentence Transformer:
    Accuracy: 0.225
    F1: 0.22427065954133768
    MAP@3: 0.3825
RoBERTa:
    Accuracy: 0.225
    F1: 0.22427065954133768
    MAP@3: 0.9429166666666666


# Prediction on test data with highest MAP@3 model

In [22]:
# RoBERTa has the highest MAP@3 score
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
test["cleaned_prompt"] = test["prompt"].apply(clean_text)
for opt in ["A", "B", "C", "D", "E"]:
    test[f"cleaned_{opt}"] = test[opt].apply(clean_text)
    
rows = []
for _, row in test.iterrows():
    prompt = row["cleaned_prompt"]
    for opt in options:
        rows.append({"question_id": row["id"],
                    "text": prompt + " </s> " + row[f"cleaned_{opt}"],
                    "option": opt})
        
test_pairs = pd.DataFrame(rows)
test_dataset = Dataset.from_pandas(test_pairs)
test_dataset = test_dataset.map(tokenize_function, batched=True)
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

preds = trainer.predict(test_dataset)
probs = torch.softmax(torch.tensor(preds.predictions),dim=1).numpy()
positive_probs = probs[:, 1]
predictions = []
for i in range(0, len(positive_probs), rows_per_q):
    q_probs = positive_probs[i:i+5]
    ranked = np.argsort(q_probs)[::-1]
    top3 = [options[j] for j in ranked[:3]]
    predictions.append(" ".join(top3))
    
submission = pd.DataFrame({"id": test["id"], "prediction": predictions})
submission.to_csv("submission.csv", index=False)

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [23]:
submission.head()

,id,prediction
0,1,A C B
1,2,B D E
2,3,B E D
3,4,E C D
4,5,C A D


In [ ]:
# sample= pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
# sample.to_csv('submission.csv', index=False)